# Task 10: User Authentication with Flask-Login

**Goal:** Implement user authentication and session management using Flask-Login.

This notebook creates a small Flask web app with:

- user registration
- login
- logout
- protected dashboard page
- password hashing
- session management using Flask-Login

**Main tools used:**
- Flask
- Flask-Login
- Flask-SQLAlchemy
- Werkzeug password hashing

In [ ]:
# Install required libraries
!pip install flask flask-login flask-sqlalchemy werkzeug

## Step 1: Create Flask app file

This code creates `flask_login_app.py`.

In [ ]:
%%writefile flask_login_app.py
from flask import Flask, render_template_string, request, redirect, url_for, flash
from flask_sqlalchemy import SQLAlchemy
from flask_login import LoginManager, UserMixin, login_user, logout_user, login_required, current_user
from werkzeug.security import generate_password_hash, check_password_hash

# --------------------------------------------------
# Flask app configuration
# --------------------------------------------------
app = Flask(__name__)

# Secret key is required for sessions and flash messages.
# In production, keep this secret and do not hardcode it.
app.config["SECRET_KEY"] = "dev-secret-key-for-flask-login"

# SQLite database file
app.config["SQLALCHEMY_DATABASE_URI"] = "sqlite:///users.db"
app.config["SQLALCHEMY_TRACK_MODIFICATIONS"] = False

# Initialize database
db = SQLAlchemy(app)

# Initialize Flask-Login
login_manager = LoginManager()
login_manager.init_app(app)

# If user tries to access protected page without login,
# Flask-Login will redirect them to login page.
login_manager.login_view = "login"


# --------------------------------------------------
# User model
# --------------------------------------------------
class User(UserMixin, db.Model):
    """
    UserMixin gives default Flask-Login properties:
    is_authenticated, is_active, is_anonymous, get_id()
    """
    id = db.Column(db.Integer, primary_key=True)
    username = db.Column(db.String(80), unique=True, nullable=False)
    email = db.Column(db.String(120), unique=True, nullable=False)
    password_hash = db.Column(db.String(200), nullable=False)

    def set_password(self, password):
        # Store hashed password, not plain password
        self.password_hash = generate_password_hash(password)

    def check_password(self, password):
        # Compare entered password with stored hashed password
        return check_password_hash(self.password_hash, password)


# --------------------------------------------------
# Flask-Login user loader
# --------------------------------------------------
@login_manager.user_loader
def load_user(user_id):
    """
    Flask-Login uses this function to load user from session.
    """
    return User.query.get(int(user_id))


# --------------------------------------------------
# HTML templates as strings
# For a bigger project, keep templates in separate .html files.
# --------------------------------------------------
base_style = """
<style>
    body {
        font-family: Arial, sans-serif;
        margin: 40px;
        background: #f4f6f8;
    }
    .box {
        background: white;
        padding: 25px;
        max-width: 500px;
        margin: auto;
        border-radius: 10px;
        box-shadow: 0 2px 10px rgba(0,0,0,0.1);
    }
    input {
        width: 100%;
        padding: 10px;
        margin: 8px 0;
    }
    button {
        padding: 10px 16px;
        cursor: pointer;
    }
    a {
        text-decoration: none;
    }
    .message {
        color: red;
    }
</style>
"""

home_template = base_style + """
<div class="box">
    <h2>Flask-Login Authentication App</h2>
    <p>This is the home page.</p>

    {% if current_user.is_authenticated %}
        <p>You are logged in as <b>{{ current_user.username }}</b>.</p>
        <a href="{{ url_for('dashboard') }}">Go to Dashboard</a><br><br>
        <a href="{{ url_for('logout') }}">Logout</a>
    {% else %}
        <a href="{{ url_for('register') }}">Register</a><br><br>
        <a href="{{ url_for('login') }}">Login</a>
    {% endif %}
</div>
"""

register_template = base_style + """
<div class="box">
    <h2>Register</h2>

    {% with messages = get_flashed_messages() %}
        {% if messages %}
            {% for message in messages %}
                <p class="message">{{ message }}</p>
            {% endfor %}
        {% endif %}
    {% endwith %}

    <form method="POST">
        <label>Username</label>
        <input type="text" name="username" required>

        <label>Email</label>
        <input type="email" name="email" required>

        <label>Password</label>
        <input type="password" name="password" required>

        <button type="submit">Register</button>
    </form>

    <p>Already have an account? <a href="{{ url_for('login') }}">Login here</a></p>
</div>
"""

login_template = base_style + """
<div class="box">
    <h2>Login</h2>

    {% with messages = get_flashed_messages() %}
        {% if messages %}
            {% for message in messages %}
                <p class="message">{{ message }}</p>
            {% endfor %}
        {% endif %}
    {% endwith %}

    <form method="POST">
        <label>Email</label>
        <input type="email" name="email" required>

        <label>Password</label>
        <input type="password" name="password" required>

        <button type="submit">Login</button>
    </form>

    <p>No account? <a href="{{ url_for('register') }}">Register here</a></p>
</div>
"""

dashboard_template = base_style + """
<div class="box">
    <h2>Protected Dashboard</h2>
    <p>Welcome, <b>{{ current_user.username }}</b>!</p>
    <p>Your email is: {{ current_user.email }}</p>

    <p>This page is protected. Only logged-in users can see it.</p>

    <a href="{{ url_for('logout') }}">Logout</a>
</div>
"""


# --------------------------------------------------
# Routes
# --------------------------------------------------
@app.route("/")
def home():
    return render_template_string(home_template)


@app.route("/register", methods=["GET", "POST"])
def register():
    if request.method == "POST":
        username = request.form.get("username")
        email = request.form.get("email")
        password = request.form.get("password")

        # Check if username or email already exists
        existing_user = User.query.filter(
            (User.username == username) | (User.email == email)
        ).first()

        if existing_user:
            flash("Username or email already exists.")
            return redirect(url_for("register"))

        # Create new user
        new_user = User(username=username, email=email)
        new_user.set_password(password)

        db.session.add(new_user)
        db.session.commit()

        flash("Registration successful. Please login.")
        return redirect(url_for("login"))

    return render_template_string(register_template)


@app.route("/login", methods=["GET", "POST"])
def login():
    if request.method == "POST":
        email = request.form.get("email")
        password = request.form.get("password")

        user = User.query.filter_by(email=email).first()

        # Check if user exists and password is correct
        if user and user.check_password(password):
            login_user(user)
            return redirect(url_for("dashboard"))

        flash("Invalid email or password.")
        return redirect(url_for("login"))

    return render_template_string(login_template)


@app.route("/dashboard")
@login_required
def dashboard():
    # This route is protected.
    # If user is not logged in, they will be redirected to login page.
    return render_template_string(dashboard_template)


@app.route("/logout")
@login_required
def logout():
    logout_user()
    flash("You have been logged out.")
    return redirect(url_for("login"))


# --------------------------------------------------
# Create database and run app
# --------------------------------------------------
if __name__ == "__main__":
    with app.app_context():
        db.create_all()

    app.run(debug=True, host="0.0.0.0", port=5000)

## Step 2: Run the Flask app

Run the next cell and open:

`http://127.0.0.1:5000`

In the browser, test:
- Register
- Login
- Dashboard
- Logout

In [ ]:
# Run the Flask authentication app.
# Stop the cell when you are finished testing.
!python flask_login_app.py

## Expected Output

The app should have:

1. Home page  
2. Register page  
3. Login page  
4. Protected dashboard  
5. Logout option  

The dashboard cannot be opened unless the user is logged in.  
This shows that Flask-Login is managing user sessions properly.

## Important Explanation

`Flask-Login` handles session management.  
After login, it remembers the user using a secure session.  
The `@login_required` decorator protects routes from unauthorized users.  
Passwords are stored in hashed form using Werkzeug, which is safer than storing plain text passwords.